# 02 — Market Distributions

**Purpose:** characterize the empirical distributions, tails, cross-asset heterogeneity, and temporal persistence of the **futures / market-state feature block** before any regime model is fitted.

This notebook is deliberately diagnostic. It:

- opens `research.duckdb` in **read-only** mode;
- uses `time_series_feature_state` as the primary research table;
- analyzes returns, realized volatility, activity/flow, price-bar structure, open interest, basis/spreads, and the standard L2 state;
- compares pooled distributions with **per-underlying** distributions;
- separates **full-range** views from **central-range** views instead of clipping or winsorizing observations;
- inspects extreme observations in their original timestamp / asset context;
- studies autocorrelation and session-to-session variation;
- evaluates candidate transforms only as **research hypotheses**;
- audits futures quote integrity and open-interest jumps on raw collector fields (Part I).

It does **not**:

- impute missing values;
- winsorize or delete tail events (integrity flags describe suspect states; they do not remove them);
- globally standardize across assets;
- fit HMMs / HSMMs;
- fit the Clean/Noisy latent chain;
- analyze the newly collected L2-instability block in depth (reserved for Notebook 04);
- analyze option-surface distributions (reserved for Notebook 03).

> Research rule: distribution diagnostics should reveal the data-generating structure, not force every feature to look Gaussian.

## 1. Environment and read-only database connection

The notebook follows the same database discipline as `01_feature_inventory.ipynb`: the research database is opened read-only and no production relation is modified.

The DuckDB session time zone is set to **UTC**. Without it, `TIMESTAMPTZ` values come back in the PC's local zone (Europe/Madrid, which also changes at DST), and every hour-of-day result would be shifted.


In [ ]:
from pathlib import Path
import os
import math
import warnings

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 220)
pd.set_option("display.width", 200)

load_dotenv()

data_root_env = os.getenv("DERIBIT_DATA_ROOT")
if not data_root_env:
    raise RuntimeError(
        "DERIBIT_DATA_ROOT is not defined in the environment/.env file."
    )

DATA_ROOT = Path(data_root_env).expanduser().resolve()
DB_PATH = DATA_ROOT / "database" / "research.duckdb"

if not DB_PATH.exists():
    raise FileNotFoundError(f"Research database not found: {DB_PATH}")

con = duckdb.connect(str(DB_PATH), read_only=True)
con.execute("SET TimeZone = 'UTC'")

print(f"Research DB: {DB_PATH}")
print("Connection mode: read-only")
print("Session time zone:", con.execute("SELECT current_setting('TimeZone')").fetchone()[0])

## 2. Load the market-facing state table

The table grain is expected to remain:

**one row = one underlying × one five-minute state**

Unlike Notebook 01, this notebook does **not** apply one global readiness filter. Each feature family is analyzed using its own valid observations and, where appropriate, its component-specific readiness flag. This avoids discarding valid futures observations merely because an unrelated options component was unavailable.

In [ ]:
features = con.execute(
    """
    SELECT *
    FROM time_series_feature_state
    ORDER BY state_time, underlying
    """
).df()

features["state_time"] = pd.to_datetime(features["state_time"], utc=True)

print(f"Rows:        {len(features):,}")
print(f"Columns:     {features.shape[1]:,}")
print(f"Underlyings: {features['underlying'].nunique(dropna=True):,}")
print(f"First state: {features['state_time'].min()}")
print(f"Last state:  {features['state_time'].max()}")

assert features["state_id"].duplicated().sum() == 0
assert features.duplicated(subset=["underlying", "state_time"]).sum() == 0

## 3. Analysis feature families

The groups below are intentionally narrower than the full feature inventory.

The aim is to understand the **market-state block** that may later contribute to:

- structural HSMM emissions;
- persistence / transition predictors;
- model-time normalization decisions.

Option-surface features are deferred to Notebook 03. The new L2-instability scalars are intentionally not a main subject here because their history is still short.

In [ ]:
FEATURE_GROUPS = {
    "returns": [
        "return_5m",
        "return_15m",
        "return_30m",
        "return_1h",
    ],
    "realized_volatility": [
        "realized_vol_30m",
        "realized_vol_1h",
        "realized_vol_4h",
        "realized_vol_30m_annualized",
        "realized_vol_1h_annualized",
        "realized_vol_4h_annualized",
    ],
    "activity_flow": [
        "volume_usd",
        "log1p_volume_usd",
        "signed_volume_usd",
        "trade_flow_imbalance",
        "buy_volume_share",
        "average_trade_size_usd",
        "trade_count",
    ],
    "bar_structure_open_interest": [
        "intrabar_range_bps",
        "close_vs_vwap_bps",
        "open_interest",
        "open_interest_change_5m",
        "open_interest_return_5m",
    ],
    "basis_spread_microprice": [
        "futures_spread_bps",
        "perpetual_mid_vs_index_bps",
        "perpetual_mark_vs_index_bps",
        "microprice_vs_mid_bps",
        "book_twa_spread_bps",
        "book_twa_microprice_vs_reference_bps",
    ],
    "standard_l2": [
        "book_twa_obi_5",
        "book_twa_obi_10",
        "book_twa_obi_20",
        "book_twa_obi_20_change_5m",
        "book_twa_bid_depth_20",
        "book_twa_ask_depth_20",
        "book_twa_spread",
        "book_twa_microprice",
    ],
}

available_groups = {
    group: [c for c in cols if c in features.columns]
    for group, cols in FEATURE_GROUPS.items()
}

for group, cols in available_groups.items():
    print(f"{group:<30} {len(cols):>2} / {len(FEATURE_GROUPS[group]):>2} available")
    missing = [c for c in FEATURE_GROUPS[group] if c not in features.columns]
    if missing:
        print("  missing:", missing)

## 4. Generic diagnostic helpers

Two principles are built into these helpers:

1. **Full range and central range are different questions.** Full-range plots show the existence of tail events; central-range plots show ordinary-state structure.
2. **Pooled crypto data can be misleading.** Per-underlying summaries remain first-class outputs throughout this notebook.

In [ ]:
CENTRAL_LOWER_Q = 0.005
CENTRAL_UPPER_Q = 0.995
TAIL_QS = [0.001, 0.01, 0.05, 0.50, 0.95, 0.99, 0.999]

def clean_numeric(df, column):
    return (
        pd.to_numeric(df[column], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

def feature_summary(df, columns, group_col=None):
    rows = []
    iterator = [("ALL", df)] if group_col is None else df.groupby(group_col, dropna=False)

    for group_name, group in iterator:
        for col in columns:
            if col not in group.columns:
                continue

            s = clean_numeric(group, col)
            if s.empty:
                continue

            q = s.quantile(TAIL_QS)
            rows.append({
                "group": group_name,
                "feature": col,
                "n": len(s),
                "missing_pct": 100.0 * (len(group) - len(s)) / len(group),
                "mean": s.mean(),
                "std": s.std(),
                "skew": s.skew(),
                "excess_kurtosis": s.kurt(),
                "min": s.min(),
                "q001": q.loc[0.001],
                "q01": q.loc[0.01],
                "q05": q.loc[0.05],
                "median": q.loc[0.50],
                "q95": q.loc[0.95],
                "q99": q.loc[0.99],
                "q999": q.loc[0.999],
                "max": s.max(),
            })

    return pd.DataFrame(rows)

def plot_full_and_central(df, column, bins=100):
    s = clean_numeric(df, column)
    if s.empty:
        print(f"{column}: no observations")
        return

    lo = s.quantile(CENTRAL_LOWER_Q)
    hi = s.quantile(CENTRAL_UPPER_Q)
    central = s[(s >= lo) & (s <= hi)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    axes[0].hist(s, bins=bins)
    axes[0].set_title(f"{column} — full range")
    axes[0].set_ylabel("Count")

    axes[1].hist(central, bins=bins)
    axes[1].set_title(
        f"{column} — central {100*(CENTRAL_UPPER_Q-CENTRAL_LOWER_Q):.1f}%"
    )
    axes[1].set_ylabel("Count")

    plt.tight_layout()
    plt.show()

    print(
        f"n={len(s):,} | "
        f"central bounds [{lo:.6g}, {hi:.6g}] | "
        f"min={s.min():.6g} | max={s.max():.6g}"
    )

def plot_asset_histograms(df, column, bins=70):
    assets = sorted(df["underlying"].dropna().unique().tolist())
    if not assets:
        return

    ncols = 2
    nrows = math.ceil(len(assets) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.7 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, asset in zip(axes, assets):
        s = clean_numeric(df[df["underlying"] == asset], column)
        if s.empty:
            ax.text(0.5, 0.5, "No observations", ha="center", va="center")
        else:
            lo = s.quantile(CENTRAL_LOWER_Q)
            hi = s.quantile(CENTRAL_UPPER_Q)
            central = s[(s >= lo) & (s <= hi)]
            ax.hist(central, bins=bins)
        ax.set_title(str(asset))

    for ax in axes[len(assets):]:
        ax.axis("off")

    fig.suptitle(f"{column} — per-asset central distributions", y=1.01)
    plt.tight_layout()
    plt.show()

def plot_ecdf_by_asset(df, column):
    fig, ax = plt.subplots(figsize=(10, 6))

    plotted = 0
    for asset, group in df.groupby("underlying"):
        s = clean_numeric(group, column).sort_values()
        if s.empty:
            continue

        y = np.arange(1, len(s) + 1) / len(s)
        ax.plot(s.to_numpy(), y, label=str(asset), linewidth=1.3)
        plotted += 1

    if plotted:
        ax.set_title(f"ECDF by underlying — {column}")
        ax.set_xlabel(column)
        ax.set_ylabel("Empirical CDF")
        ax.legend()
        plt.tight_layout()
        plt.show()

def contiguous_acf(df, column, max_lag=24):
    rows = []

    for asset, group in df.groupby("underlying"):
        g = group.sort_values("state_time").copy()
        x = pd.to_numeric(g[column], errors="coerce")

        for lag in range(1, max_lag + 1):
            lagged_x = x.shift(lag)
            lagged_time = g["state_time"].shift(lag)

            expected_minutes = 5 * lag
            valid = (
                x.notna()
                & lagged_x.notna()
                & (
                    (g["state_time"] - lagged_time)
                    .dt.total_seconds()
                    .div(60.0)
                    == expected_minutes
                )
            )

            n_pairs = int(valid.sum())
            corr = np.nan

            if n_pairs >= 20:
                corr = x[valid].corr(lagged_x[valid])

            rows.append({
                "underlying": asset,
                "feature": column,
                "lag": lag,
                "lag_minutes": expected_minutes,
                "n_pairs": n_pairs,
                "acf": corr,
            })

    return pd.DataFrame(rows)

def plot_acf_by_asset(df, column, max_lag=24):
    acf = contiguous_acf(df, column, max_lag=max_lag)

    fig, ax = plt.subplots(figsize=(11, 6))
    for asset, group in acf.groupby("underlying"):
        ax.plot(group["lag_minutes"], group["acf"], marker="o", ms=3, label=str(asset))

    ax.axhline(0.0, linewidth=1)
    ax.set_title(f"Autocorrelation by underlying — {column}")
    ax.set_xlabel("Lag (minutes)")
    ax.set_ylabel("Autocorrelation")
    ax.legend()
    plt.tight_layout()
    plt.show()

    return acf

def contextual_extremes(df, column, n=12, absolute=False):
    context = [
        "state_id",
        "state_time",
        "underlying",
        "expiry_date",
        "minutes_to_expiry",
        column,
        "return_5m",
        "realized_vol_1h_annualized",
        "volume_usd",
        "trade_flow_imbalance",
        "futures_spread_bps",
        "perpetual_mid_vs_index_bps",
        "book_twa_obi_20",
        "book_twa_spread_bps",
    ]
    context = list(dict.fromkeys(c for c in context if c in df.columns))

    work = df[context].copy()
    work[column] = pd.to_numeric(work[column], errors="coerce")
    work = work.dropna(subset=[column])

    if absolute:
        work["_rank_value"] = work[column].abs()
        result = work.nlargest(n, "_rank_value").drop(columns="_rank_value")
    else:
        high = work.nlargest(n, column).assign(tail="high")
        low = work.nsmallest(n, column).assign(tail="low")
        result = pd.concat([low, high], ignore_index=True)

    return result.sort_values(
        ["state_time", "underlying"],
        kind="stable"
    ).reset_index(drop=True)

def signed_log1p(s):
    s = pd.to_numeric(s, errors="coerce")
    return np.sign(s) * np.log1p(np.abs(s))

## 5. Feature availability by asset

Notebook 01 established overall integrity. Here we verify that the effective sample size of each **market** feature is understood before comparing distribution shape.

A feature with a smaller sample is not automatically defective; rolling windows and component readiness naturally reduce coverage.

In [ ]:
market_feature_columns = sorted({
    c
    for cols in available_groups.values()
    for c in cols
})

availability_rows = []

for asset, group in features.groupby("underlying"):
    row = {
        "underlying": asset,
        "states": len(group),
    }

    for col in market_feature_columns:
        row[f"{col}_n"] = int(group[col].notna().sum())
        row[f"{col}_pct"] = 100.0 * group[col].notna().mean()

    availability_rows.append(row)

availability_by_asset = pd.DataFrame(availability_rows).sort_values("underlying")
display(availability_by_asset)

overall_availability = pd.DataFrame({
    "feature": market_feature_columns,
    "n": [int(features[c].notna().sum()) for c in market_feature_columns],
    "coverage_pct": [100.0 * features[c].notna().mean() for c in market_feature_columns],
}).sort_values("coverage_pct")

display(overall_availability.reset_index(drop=True))

# Part A — Returns

## 6. Return distributions

**Source.** From market feature v0.2.0, `return_5m`–`return_1h` and all realized-volatility columns are computed from the **Deribit index** (`return_source = 'deribit_index'`), the settlement reference of the options. The perpetual-mid return survives only as the diagnostic `perp_mid_return_5m` (Section 30 shows why).

The return block is examined without forcing normality. Heavy tails, asymmetry, and asset-specific scale are economically meaningful diagnostics for later emission-family choices.

The four return horizons overlap heavily, so they should not be interpreted as independent observations.

In [ ]:
return_cols = available_groups["returns"]

return_pooled = feature_summary(features, return_cols)
return_by_asset = feature_summary(features, return_cols, group_col="underlying")

display(return_pooled)
display(return_by_asset)

for col in return_cols:
    plot_full_and_central(features, col)

## 7. Per-asset return shape and empirical CDF

Pooled returns can hide substantial differences between BTC, ETH, and smaller underlyings. The ECDF provides a distribution comparison that does not depend on histogram bin choices.

In [ ]:
if "return_5m" in features.columns:
    plot_asset_histograms(features, "return_5m")
    plot_ecdf_by_asset(features, "return_5m")

if "return_1h" in features.columns:
    plot_asset_histograms(features, "return_1h")
    plot_ecdf_by_asset(features, "return_1h")

## 8. Absolute-return tails

For structural-regime research, the magnitude process can be as important as the signed return itself.

This section compares absolute 5-minute returns by underlying and lists the largest observed events without removing them.

In [ ]:
if "return_5m" in features.columns:
    abs_return_rows = []

    for asset, group in features.groupby("underlying"):
        s = clean_numeric(group, "return_5m").abs()
        if s.empty:
            continue

        q = s.quantile([0.50, 0.90, 0.95, 0.99, 0.999])

        abs_return_rows.append({
            "underlying": asset,
            "n": len(s),
            "median_abs_return": q.loc[0.50],
            "q90_abs_return": q.loc[0.90],
            "q95_abs_return": q.loc[0.95],
            "q99_abs_return": q.loc[0.99],
            "q999_abs_return": q.loc[0.999],
            "max_abs_return": s.max(),
        })

    display(pd.DataFrame(abs_return_rows).sort_values("q99_abs_return", ascending=False))

    print("Largest absolute 5-minute returns:")
    display(contextual_extremes(features, "return_5m", n=25, absolute=True))

## 9. Return autocorrelation

Short-horizon return autocorrelation is checked per asset on the actual five-minute grid.

This diagnostic can later help distinguish low directional persistence, momentum-like persistence, anti-persistence / mean-reversion signatures, and microstructure effects. It is **not** yet a regime label.

In [ ]:
return_acf = {}

for col in [c for c in ["return_5m", "return_15m"] if c in features.columns]:
    return_acf[col] = plot_acf_by_asset(features, col, max_lag=24)

if "return_5m" in return_acf:
    display(return_acf["return_5m"].head(30))

# Part B — Realized Volatility

## 10. Realized-volatility distributions and effective sample size

Realized volatility is right-skewed by construction and its horizons have different readiness requirements. Sample size is therefore reported alongside every comparison.

Both raw-window and annualized forms are retained. Annualization changes scale, not information content.

In [ ]:
rv_cols = available_groups["realized_volatility"]

rv_summary = feature_summary(features, rv_cols)
rv_by_asset = feature_summary(features, rv_cols, group_col="underlying")

display(rv_summary)
display(rv_by_asset)

for col in [
    c for c in [
        "realized_vol_30m_annualized",
        "realized_vol_1h_annualized",
        "realized_vol_4h_annualized",
    ]
    if c in features.columns
]:
    plot_full_and_central(features, col)

## 11. Volatility persistence

Persistent volatility is a plausible structural-state input. We therefore inspect autocorrelation independently for each underlying.

Longer-horizon realized-volatility features mechanically overlap, so high autocorrelation is expected. The question is **how persistent and how asset-dependent** it is, not whether the ACF is statistically zero.

In [ ]:
rv_acf = {}

for col in [
    c for c in [
        "realized_vol_30m_annualized",
        "realized_vol_1h_annualized",
        "realized_vol_4h_annualized",
    ]
    if c in features.columns
]:
    rv_acf[col] = plot_acf_by_asset(features, col, max_lag=48)

# Part C — Activity and Trade Flow

## 12. Volume, trade size, and flow distributions

The activity block contains several fundamentally different supports:

- volume and trade size are non-negative and typically long-tailed;
- signed volume is unbounded in both directions;
- trade-flow imbalance lies in `[-1, 1]`;
- buy-volume share lies in `[0, 1]`.

Boundary masses at `-1`, `0`, or `+1` are treated as real features of the data rather than as histogram artifacts.

In [ ]:
activity_cols = available_groups["activity_flow"]

display(feature_summary(features, activity_cols))
display(feature_summary(features, activity_cols, group_col="underlying"))

for col in [
    c for c in [
        "volume_usd",
        "log1p_volume_usd",
        "signed_volume_usd",
        "trade_flow_imbalance",
        "buy_volume_share",
        "average_trade_size_usd",
    ]
    if c in features.columns
]:
    plot_full_and_central(features, col)

## 13. Structural probability masses: zero volume and ±1 flow

Extreme flow imbalance can occur when all observed aggressor flow in a window is one-sided. That is not equivalent to a smooth continuous observation close to ±1, so the endpoint mass is measured explicitly.

In [ ]:
boundary_rows = []

for asset, group in features.groupby("underlying"):
    row = {"underlying": asset, "states": len(group)}

    if "volume_usd" in group.columns:
        vol = pd.to_numeric(group["volume_usd"], errors="coerce")
        valid = vol.notna()
        row["volume_valid_n"] = int(valid.sum())
        row["volume_zero_pct_of_valid"] = (
            100.0 * (vol[valid] == 0).mean() if valid.any() else np.nan
        )

    if "trade_flow_imbalance" in group.columns:
        flow = pd.to_numeric(group["trade_flow_imbalance"], errors="coerce")
        valid = flow.notna()
        row["flow_valid_n"] = int(valid.sum())
        row["flow_at_minus_1_pct"] = (
            100.0 * np.isclose(flow[valid], -1.0).mean() if valid.any() else np.nan
        )
        row["flow_at_plus_1_pct"] = (
            100.0 * np.isclose(flow[valid], 1.0).mean() if valid.any() else np.nan
        )
        row["flow_at_either_boundary_pct"] = (
            100.0
            * (
                np.isclose(flow[valid], -1.0)
                | np.isclose(flow[valid], 1.0)
            ).mean()
            if valid.any()
            else np.nan
        )

    if "buy_volume_share" in group.columns:
        share = pd.to_numeric(group["buy_volume_share"], errors="coerce")
        valid = share.notna()
        row["buy_share_zero_pct"] = (
            100.0 * np.isclose(share[valid], 0.0).mean() if valid.any() else np.nan
        )
        row["buy_share_one_pct"] = (
            100.0 * np.isclose(share[valid], 1.0).mean() if valid.any() else np.nan
        )

    boundary_rows.append(row)

boundary_mass = pd.DataFrame(boundary_rows).sort_values("underlying")
display(boundary_mass)

## 14. Activity persistence

Volume and trade intensity often cluster. Flow direction may behave very differently.

That distinction matters for the later structural/noise split: a highly persistent activity state is not the same phenomenon as rapidly alternating signed flow.

In [ ]:
for col in [
    c for c in [
        "log1p_volume_usd",
        "trade_flow_imbalance",
        "average_trade_size_usd",
    ]
    if c in features.columns
]:
    plot_acf_by_asset(features, col, max_lag=24)

# Part D — Price-Bar Structure and Open Interest

## 15. Intrabar range, VWAP relation, and open-interest changes

`close_vs_vwap_bps` is the close relative to the **five-minute bar VWAP**. It is not yet the longer-horizon/session VWAP displacement envisioned for later structural research.

Open-interest changes are kept in both level-change and proportional-return form because asset scale may matter.

In [ ]:
bar_oi_cols = available_groups["bar_structure_open_interest"]

display(feature_summary(features, bar_oi_cols))
display(feature_summary(features, bar_oi_cols, group_col="underlying"))

for col in [
    c for c in [
        "intrabar_range_bps",
        "close_vs_vwap_bps",
        "open_interest_change_5m",
        "open_interest_return_5m",
    ]
    if c in features.columns
]:
    plot_full_and_central(features, col)

In [ ]:
for col in [
    c for c in [
        "intrabar_range_bps",
        "close_vs_vwap_bps",
        "open_interest_return_5m",
    ]
    if c in features.columns
]:
    plot_acf_by_asset(features, col, max_lag=24)

# Part E — Basis, Spread, and Microprice

## 16. Distribution diagnostics

Notebook 01 exposed unusually large maxima in several spread/basis features relative to their ordinary quantiles. Here the goal is to determine whether these are concentrated in one asset, concentrated near particular session times, associated with high volatility / volume, associated with wider L2 spreads, or isolated observations versus repeated episodes.

No observation is deleted in this notebook.

In [ ]:
basis_cols = available_groups["basis_spread_microprice"]

display(feature_summary(features, basis_cols))
display(feature_summary(features, basis_cols, group_col="underlying"))

for col in basis_cols:
    plot_full_and_central(features, col)

## 17. Contextual tail inspection

The following tables identify the actual observations behind the tails. This is more informative than simply reporting skewness or kurtosis.

For signed basis/microprice features, both tails are shown. For non-negative spread features, the largest observations are emphasized.

In [ ]:
extreme_features = [
    c for c in [
        "futures_spread_bps",
        "perpetual_mid_vs_index_bps",
        "perpetual_mark_vs_index_bps",
        "microprice_vs_mid_bps",
        "book_twa_spread_bps",
        "book_twa_microprice_vs_reference_bps",
    ]
    if c in features.columns
]

for col in extreme_features:
    print("\n" + "=" * 110)
    print(col)
    print("=" * 110)

    if col in {"futures_spread_bps", "book_twa_spread_bps"}:
        display(contextual_extremes(features, col, n=20, absolute=True))
    else:
        display(contextual_extremes(features, col, n=12, absolute=False))

# Part F — Standard L2 State

## 18. Persistent order-book state

This section covers the established L2 level features with sufficient history:

- time-weighted OBI at 5/10/20 levels;
- 5-minute change in deep-book imbalance;
- time-weighted depth;
- time-weighted spread / microprice.

The newer streaming instability statistics (`book_obi_20_std`, roughness, coherence noise, reversal-strength distributions, etc.) are deferred to **Notebook 04**, where their shorter history can be handled explicitly.

In [ ]:
l2_cols = available_groups["standard_l2"]

display(feature_summary(features, l2_cols))
display(feature_summary(features, l2_cols, group_col="underlying"))

for col in [
    c for c in [
        "book_twa_obi_5",
        "book_twa_obi_10",
        "book_twa_obi_20",
        "book_twa_obi_20_change_5m",
        "book_twa_bid_depth_20",
        "book_twa_ask_depth_20",
        "book_twa_spread",
    ]
    if c in features.columns
]:
    plot_full_and_central(features, col)

## 19. Diagnostic depth imbalance

The roadmap calls for depth-ratio style features. The production feature table currently preserves bid and ask depth separately.

For research only, this notebook constructs a diagnostic bounded depth imbalance:

\[
\frac{BidDepth_{20} - AskDepth_{20}}
     {BidDepth_{20} + AskDepth_{20}}
\]

This does **not** promote the variable into the production feature contract. It only tests whether this scale-free representation may be useful later.

**Redundancy check.** `book_twa_obi_20` is the time-weighted average of the instantaneous imbalance, while this variable is the imbalance of time-weighted depths. They differ only through the covariance of depth and time weights, so they are expected to be near-duplicates. The cell reports their correlation; if it is close to 1, the diagnostic adds nothing and is not carried into the persistence study.


In [ ]:
if {
    "book_twa_bid_depth_20",
    "book_twa_ask_depth_20",
}.issubset(features.columns):

    denom = (
        pd.to_numeric(features["book_twa_bid_depth_20"], errors="coerce")
        + pd.to_numeric(features["book_twa_ask_depth_20"], errors="coerce")
    )

    features["_diag_depth_imbalance_20"] = np.where(
        denom > 0,
        (
            pd.to_numeric(features["book_twa_bid_depth_20"], errors="coerce")
            - pd.to_numeric(features["book_twa_ask_depth_20"], errors="coerce")
        ) / denom,
        np.nan,
    )

    display(
        feature_summary(
            features,
            ["_diag_depth_imbalance_20"],
            group_col="underlying"
        )
    )

    plot_full_and_central(features, "_diag_depth_imbalance_20")
    plot_asset_histograms(features, "_diag_depth_imbalance_20")

    if "book_twa_obi_20" in features.columns:
        pair = pd.concat(
            [
                pd.to_numeric(features["book_twa_obi_20"], errors="coerce"),
                features["_diag_depth_imbalance_20"],
            ],
            axis=1,
        ).dropna()
        print("Correlation with book_twa_obi_20 by underlying:")
        display(
            features.loc[pair.index]
            .groupby("underlying")
            .apply(lambda g: g["book_twa_obi_20"].astype(float).corr(g["_diag_depth_imbalance_20"]))
            .to_frame("corr")
            .round(4)
        )
        print(f"Max |difference|: {(pair.iloc[:, 0] - pair.iloc[:, 1]).abs().max():.4f}")


## 20. L2 persistence

Deep-book imbalance is particularly relevant to the structural-regime hypothesis because a persistent directional liquidity imbalance may support momentum, while unstable or rapidly reversing imbalance may belong to the later noise/transition block.

Here we study only the persistent level and its five-minute change.

In [ ]:
for col in [
    c for c in [
        "book_twa_obi_20",
        "book_twa_obi_20_change_5m",
    ]
    if c in features.columns
]:
    plot_acf_by_asset(features, col, max_lag=36)

# Part G — Cross-Asset Scale and Session Variation

## 21. Cross-asset scale comparison

We intentionally do **not** standardize here. The purpose is to expose where asset scale differs enough that a later model may need asset-conditioned centering/scaling, robust scale estimates, relative/bounded representations, or separate treatment by feature family.

The table reports medians, IQRs, and 99th-percentile absolute magnitudes for a compact set of representative features.

In [ ]:
scale_features = [
    c for c in [
        "return_5m",
        "realized_vol_1h_annualized",
        "log1p_volume_usd",
        "trade_flow_imbalance",
        "intrabar_range_bps",
        "open_interest_return_5m",
        "futures_spread_bps",
        "perpetual_mid_vs_index_bps",
        "book_twa_obi_20",
        "book_twa_obi_20_change_5m",
    ]
    if c in features.columns
]

scale_rows = []

for asset, group in features.groupby("underlying"):
    for col in scale_features:
        s = clean_numeric(group, col)
        if s.empty:
            continue

        q25, q50, q75 = s.quantile([0.25, 0.50, 0.75])

        scale_rows.append({
            "underlying": asset,
            "feature": col,
            "n": len(s),
            "median": q50,
            "iqr": q75 - q25,
            "std": s.std(),
            "q99_abs": s.abs().quantile(0.99),
            "max_abs": s.abs().max(),
        })

cross_asset_scale = pd.DataFrame(scale_rows)
display(cross_asset_scale.sort_values(["feature", "underlying"]))

## 22. Distribution drift across expiry sessions

A regime model will eventually be trained chronologically. Before that, we need to know whether feature distributions themselves move materially from session to session.

With the current short history this is only a diagnostic, not a stationarity conclusion.

In [ ]:
session_features = [
    c for c in [
        "return_5m",
        "realized_vol_1h_annualized",
        "log1p_volume_usd",
        "futures_spread_bps",
        "book_twa_obi_20",
    ]
    if c in features.columns
]

session_rows = []

for (expiry, asset), group in features.groupby(["expiry_date", "underlying"]):
    for col in session_features:
        s = clean_numeric(group, col)
        if s.empty:
            continue

        session_rows.append({
            "expiry_date": expiry,
            "underlying": asset,
            "feature": col,
            "n": len(s),
            "median": s.median(),
            "q90": s.quantile(0.90),
            "q95": s.quantile(0.95),
            "q99": s.quantile(0.99),
        })

session_distribution = pd.DataFrame(session_rows)
display(session_distribution.head(30))

for col in session_features:
    subset = session_distribution[session_distribution["feature"] == col]
    if subset.empty:
        continue

    fig, ax = plt.subplots(figsize=(12, 5))
    for asset, group in subset.groupby("underlying"):
        ax.plot(
            group["expiry_date"],
            group["median"],
            marker="o",
            ms=3,
            label=str(asset),
        )

    ax.set_title(f"Session median by asset — {col}")
    ax.set_xlabel("Expiry session")
    ax.set_ylabel("Median")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
    plt.tight_layout()
    plt.show()

# Part H — Candidate Transformations

## 23. Transformation diagnostics

Transformations are evaluated for **scale management and model stability**, not for cosmetic Gaussianization.

Candidate rules used here:

- `log1p(x)` for non-negative long-tailed variables;
- `sign(x) * log1p(|x|)` for signed scale variables;
- raw form retained for bounded variables and returns.

A reduction in skewness is useful evidence, but it is **not sufficient by itself** to promote a transform into production.

In [ ]:
TRANSFORM_SPECS = {
    "volume_usd": ("log1p", "non-negative activity scale"),
    "average_trade_size_usd": ("log1p", "non-negative trade-size scale"),
    "intrabar_range_bps": ("log1p", "non-negative range scale"),
    "futures_spread_bps": ("log1p", "non-negative spread scale"),
    "book_twa_spread_bps": ("log1p", "non-negative spread scale"),
    "realized_vol_30m_annualized": ("log1p", "positive volatility scale"),
    "realized_vol_1h_annualized": ("log1p", "positive volatility scale"),
    "realized_vol_4h_annualized": ("log1p", "positive volatility scale"),
    "signed_volume_usd": ("signed_log1p", "signed activity scale"),
    "open_interest_change_5m": ("signed_log1p", "signed level-change scale"),
    "perpetual_mid_vs_index_bps": ("signed_log1p", "signed basis scale"),
    "perpetual_mark_vs_index_bps": ("signed_log1p", "signed basis scale"),
    "microprice_vs_mid_bps": ("signed_log1p", "signed microprice displacement"),
    "book_twa_microprice_vs_reference_bps": (
        "signed_log1p",
        "signed microprice displacement"
    ),
}

transform_rows = []

for col, (transform_name, reason) in TRANSFORM_SPECS.items():
    if col not in features.columns:
        continue

    raw = pd.to_numeric(features[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

    if transform_name == "log1p":
        transformed = np.log1p(raw.where(raw >= 0))
    elif transform_name == "signed_log1p":
        transformed = signed_log1p(raw)
    else:
        transformed = raw.copy()

    valid_raw = raw.dropna()
    valid_transformed = pd.Series(transformed).replace([np.inf, -np.inf], np.nan).dropna()

    transform_rows.append({
        "feature": col,
        "candidate_transform": transform_name,
        "reason": reason,
        "raw_n": len(valid_raw),
        "raw_skew": valid_raw.skew() if len(valid_raw) else np.nan,
        "raw_excess_kurtosis": valid_raw.kurt() if len(valid_raw) else np.nan,
        "transformed_n": len(valid_transformed),
        "transformed_skew": valid_transformed.skew() if len(valid_transformed) else np.nan,
        "transformed_excess_kurtosis": (
            valid_transformed.kurt() if len(valid_transformed) else np.nan
        ),
    })

transform_assessment = pd.DataFrame(transform_rows)
display(transform_assessment)

## 24. Features that should initially remain in native bounded / return form

These are not automatic production decisions. They are the default hypotheses to carry into the later feature-contract notebook.

- Returns: retain signed return interpretation; evaluate robust emission families rather than forcing normality.
- `trade_flow_imbalance`, `buy_volume_share`, and OBI variables: retain bounded support and explicitly model endpoint / boundary behavior where relevant.
- Proportional OI change: retain raw initially unless tails prove numerically problematic.
- Cross-asset scale differences: preserve them until the deterministic normalization study determines what should be asset/TOD/TTE conditioned.

In [ ]:
native_form_candidates = [
    c for c in [
        "return_5m",
        "return_15m",
        "return_30m",
        "return_1h",
        "trade_flow_imbalance",
        "buy_volume_share",
        "book_twa_obi_5",
        "book_twa_obi_10",
        "book_twa_obi_20",
        "book_twa_obi_20_change_5m",
        "open_interest_return_5m",
        "_diag_depth_imbalance_20",
    ]
    if c in features.columns
]

native_summary = feature_summary(features, native_form_candidates)

if not native_summary.empty:
    display(
        native_summary[
            [
                "feature",
                "n",
                "skew",
                "excess_kurtosis",
                "min",
                "q01",
                "median",
                "q99",
                "max",
            ]
        ]
    )

# Part I — Futures Data Integrity

## 25. Why this part exists

Parts A–E show tail events that may be **feed artifacts rather than market behaviour** (audit A1): constant spreads with `return_5m = 0` for an hour while trading continued, a ±9.6% TRX round trip on zero volume, and 500–800 bps book spreads with no trades.

Before any return-based feature is trusted, each five-minute futures state is tested with signals that existed **before collector v0.3**:

| Test | Raw fields | What it detects |
|---|---|---|
| Wide quote | `best_bid_price`, `best_ask_price` | Spread far above the asset's own typical spread |
| Mid off index | ticker mid, `index_price` | Perpetual mid detached from the settlement index |
| Ticker vs L2 disagreement (**v0.3 rows only**) | ticker BBO vs `book_best_bid_price` / `book_best_ask_price` | One of the two feeds is stale |
| Stale BBO vs index | BBO unchanged across states while the index moved more than half the quoted spread | Quotes that a live market would have repriced |
| Frozen BBO with trades | identical BBO for ≥ `FROZEN_MIN_BARS` states while `trade_count > 0` | Trading continues against a frozen quote |
| Last trade outside BBO (**v0.3 rows only**) | `close`, `last_trade_timestamp_ms` vs boundary BBO | A trade printed shortly before the boundary outside the recorded quote |
| No L2 updates | `book_update_count = 0` with `trade_count > 0` | Dead book subscription |

**Scope.** Thresholds here are full-sample robust statistics, used only to **find and describe** episodes. They are not a production rule: the integrity gate in `quality_contract.py` must estimate its thresholds on training history only. Nothing is dropped from any table.

All timestamps in this part are **UTC** (`SET TimeZone='UTC'` in Section 1).

In [ ]:
RAW_WANTED = [
    "underlying", "bar_end", "instrument_name",
    "best_bid_price", "best_ask_price", "mid_price",
    "mark_price", "index_price", "open_interest",
    "book_valid", "book_best_bid_price", "book_best_ask_price", "book_mid_price",
    "book_update_count", "book_stats_complete", "book_age_ms",
    "ticker_age_ms", "ticker_exchange_timestamp_ms",
    "open", "high", "low", "close", "vwap",
    "volume_usd", "trade_count", "last_trade_timestamp_ms",
    "stream_complete", "stream_incomplete_reason", "connection_count",
    "collector_version",
]

def load_raw_futures():
    """Raw perpetual rows with every pre-v0.3 integrity field.

    Uses the research DB's raw view when present; otherwise reads the
    Parquet files directly. Both paths are read-only.
    """
    has_view = con.execute(
        "SELECT COUNT(*) FROM duckdb_views() WHERE view_name = 'raw_futures_5m'"
    ).fetchone()[0] > 0

    if has_view:
        source = "raw_futures_5m"
    else:
        glob = (DATA_ROOT / "futures" / "5m" / "date=*" / "futures_5m.parquet").as_posix()
        source = (
            f"read_parquet('{glob}', hive_partitioning = true, union_by_name = true)"
        )

    available = set(
        con.execute(f"DESCRIBE SELECT * FROM {source}").df()["column_name"]
    )
    selected = [c for c in RAW_WANTED if c in available]
    missing = [c for c in RAW_WANTED if c not in available]

    raw = con.execute(
        f"SELECT {', '.join(selected)} FROM {source} "
        "ORDER BY underlying, bar_end, instrument_name"
    ).df()

    return raw, source, missing

raw_futures_all, raw_source, raw_missing = load_raw_futures()

raw_futures_all["bar_end"] = pd.to_datetime(raw_futures_all["bar_end"], utc=True)

print(f"Source:            {raw_source}")
print(f"Raw rows:          {len(raw_futures_all):,}")
print(f"First / last bar:  {raw_futures_all['bar_end'].min()}  /  {raw_futures_all['bar_end'].max()}")
print(f"Fields not found:  {raw_missing if raw_missing else 'none'}")

# One perpetual per underlying is expected. Several instruments per
# underlying would make the canonical rank-by-name choice switch
# instruments, which would show up as OI and price jumps.
instrument_inventory = (
    raw_futures_all
    .groupby(["underlying", "instrument_name"], dropna=False)
    .agg(rows=("bar_end", "size"), first_bar=("bar_end", "min"), last_bar=("bar_end", "max"))
    .reset_index()
)
display(instrument_inventory)

dup_states = int(raw_futures_all.duplicated(["underlying", "bar_end"]).sum())
print(f"Duplicate (underlying, bar_end) rows: {dup_states:,}")

# Same canonical choice as build_canonical.py: first instrument_name per state.
raw_futures = (
    raw_futures_all
    .drop_duplicates(["underlying", "bar_end"], keep="first")
    .sort_values(["underlying", "bar_end"])
    .reset_index(drop=True)
)

if "collector_version" in raw_futures.columns:
    display(
        raw_futures
        .assign(collector_version=raw_futures["collector_version"].fillna("pre-v0.3 (unstamped)"))
        .groupby("collector_version")
        .agg(rows=("bar_end", "size"), first_bar=("bar_end", "min"), last_bar=("bar_end", "max"))
    )

## 26. Per-state integrity metrics

All lags are taken on the exact five-minute grid per asset; a lag across a gap is `NaN`, never bridged.

Definitions (bps of the relevant price):

$$
s_t = 10^4\,\frac{a_t - b_t}{m_t}, \qquad
\text{basis}_t = 10^4 \ln\frac{m_t}{I_t}, \qquad
\Delta^{\text{tick/L2}}_t = 10^4\,\frac{\left|m_t - m^{L2}_t\right|}{m_t}
$$

where $b_t, a_t, m_t$ are the ticker bid, ask and mid at the state boundary, $m^{L2}_t$ is the L2 book mid and $I_t$ is the Deribit index.

**Stale BBO vs index.** Let $t_0$ be the last state at which the ticker BBO changed. If the BBO is unchanged at $t$ but

$$
10^4\left|\ln\frac{I_t}{I_{t_0}}\right| > \tfrac12\, s_t,
$$

the recorded quote no longer brackets where the index says the market is.

**Last trade outside BBO.** Only trades within `LAST_TRADE_RECENCY_MS` of the boundary are tested, so that a legitimate move between the trade and the snapshot is small:

$$
o_t = 10^4\,\frac{\max\left(\text{close}_t - a_t,\; b_t - \text{close}_t,\; 0\right)}{m_t}
$$

In [ ]:
LAST_TRADE_RECENCY_MS = 30_000
FROZEN_MIN_BARS = 3

def lag_on_grid(g, column, lag=1):
    """Value `lag` five-minute states earlier, or NaN if the grid has a gap."""
    shifted = g[column].shift(lag)
    gap_ok = (g["bar_end"] - g["bar_end"].shift(lag)) == pd.Timedelta(minutes=5 * lag)
    return shifted.where(gap_ok)

def num(df, col):
    if col not in df.columns:
        return pd.Series(np.nan, index=df.index)
    return pd.to_numeric(df[col], errors="coerce")

def integrity_metrics(g):
    g = g.sort_values("bar_end").copy()

    bid, ask, index = num(g, "best_bid_price"), num(g, "best_ask_price"), num(g, "index_price")
    two_sided = (bid > 0) & (ask > 0) & (bid <= ask)
    mid = ((bid + ask) / 2).where(two_sided)
    g["tick_mid"] = mid
    g["tick_spread_bps"] = (1e4 * (ask - bid) / mid).where(two_sided)
    g["mid_vs_index_bps"] = (1e4 * np.log(mid / index)).where(index > 0)
    g["mark_vs_index_bps"] = (1e4 * np.log(num(g, "mark_price") / index)).where(index > 0)

    vwap = num(g, "vwap")
    g["vwap_vs_index_bps"] = (1e4 * np.log(vwap / index)).where((vwap > 0) & (index > 0))

    l2_bid, l2_ask = num(g, "book_best_bid_price"), num(g, "book_best_ask_price")
    l2_ok = (l2_bid > 0) & (l2_ask > 0) & (l2_bid <= l2_ask)
    g["l2_spread_bps"] = (1e4 * (l2_ask - l2_bid) / ((l2_ask + l2_bid) / 2)).where(l2_ok)
    g["tick_vs_l2_mid_bps"] = (1e4 * ((l2_bid + l2_ask) / 2 - mid).abs() / mid).where(l2_ok & two_sided)

    # BBO change tracking on the exact grid.
    g["_bid"], g["_ask"] = bid.where(two_sided), ask.where(two_sided)
    prev_bid, prev_ask = lag_on_grid(g, "_bid"), lag_on_grid(g, "_ask")
    g["bbo_unchanged"] = (g["_bid"] == prev_bid) & (g["_ask"] == prev_ask)

    # A new run starts whenever the BBO changes, the grid breaks or the quote is invalid.
    run_start = ~g["bbo_unchanged"]
    run_id = run_start.cumsum()
    g["bbo_run_bars"] = g.groupby(run_id).cumcount() + 1
    g["_index_at_change"] = index.where(run_start).groupby(run_id).transform("first")
    g["index_drift_since_bbo_change_bps"] = (
        1e4 * np.log(index / g["_index_at_change"]).abs()
    ).where(g["bbo_unchanged"])

    trades = num(g, "trade_count").fillna(0)
    g["_trades"] = trades
    g["bbo_frozen_trade_bars"] = (
        (g["bbo_unchanged"] & (trades > 0))
        .astype(int)
        .groupby(run_id)
        .cumsum()
    )

    # Last trade close vs the boundary quote, only for recent trades.
    end_ms = g["bar_end"].astype("int64") // 10**6
    recency = end_ms - num(g, "last_trade_timestamp_ms")
    close = num(g, "close")
    outside = np.maximum.reduce([close - ask, bid - close, pd.Series(0.0, index=g.index)])
    g["last_trade_recency_ms"] = recency
    g["last_trade_outside_bbo_bps"] = (1e4 * outside / mid).where(
        two_sided & (close > 0) & (recency >= 0) & (recency <= LAST_TRADE_RECENCY_MS)
    )

    g["index_return_5m"] = np.log(index / lag_on_grid(g.assign(_i=index), "_i")).where(index > 0)
    g["mid_return_5m_raw"] = np.log(mid / lag_on_grid(g.assign(_m=mid), "_m"))
    g["two_sided_ticker"] = two_sided

    return g.drop(columns=["_bid", "_ask", "_index_at_change", "_trades"])

def per_asset(df, func, **kwargs):
    """Apply a frame-level function per underlying (explicit loop: no groupby.apply)."""
    parts = [func(g, **kwargs) for _, g in df.groupby("underlying", sort=True)]
    return pd.concat(parts, ignore_index=True) if parts else df.iloc[0:0].copy()

integrity = per_asset(raw_futures, integrity_metrics)

metric_cols = [
    "tick_spread_bps", "l2_spread_bps", "tick_vs_l2_mid_bps",
    "mid_vs_index_bps", "mark_vs_index_bps", "vwap_vs_index_bps",
    "index_drift_since_bbo_change_bps", "last_trade_outside_bbo_bps",
    "ticker_age_ms", "book_age_ms", "book_update_count",
]
display(feature_summary(integrity, [c for c in metric_cols if c in integrity.columns], group_col="underlying"))

## 27. Candidate integrity flags

Robust per-asset scale: for a metric $x$ with per-asset median $\tilde x$,

$$
z_t = \frac{x_t - \tilde x}{\max\left(1.4826\,\mathrm{MAD}(x),\ \text{floor}\right)}
$$

**Why a robust z alone is not enough.** BTC, ETH and SOL quote on one tick almost all the time (BTC median spread ≈ 0.012 bps), so $\mathrm{MAD}=0$ and a 10-tick spread of 0.1 bps scores $z>8$. The first run of this notebook flagged 27% of SOL states this way, and the flagged bars clustered in volatile periods. Masking them would have removed genuine volatility.

Every level flag therefore also needs an **economic floor**:

| Flag | Fires when |
|---|---|
| Wide spread | $z_{\log s} > z^*$ **and** $s_t \ge$ `WIDE_SPREAD_MIN_BPS` **and** $s_t \ge$ `WIDE_SPREAD_MIN_RATIO` $\times \tilde s$ |
| Mid off index | $\lvert z_{\text{basis}}\rvert > z^*$ **and** $\lvert\text{basis}_t - \widetilde{\text{basis}}\rvert \ge$ `BASIS_MIN_DEVIATION_BPS` |
| Stale vs index | BBO unchanged **and** index drift $> \max\left(\tfrac12 s_t,\ \text{`STALE_DRIFT_MIN_BPS`}\right)$ |

**Pre-v0.3 coverage.** `tick_l2_disagree` and `trade_outside_bbo` need fields that only v0.3 rows carry reliably (L2 best bid/ask), and the trade test cannot catch trades inside a frozen wide quote. They are reported separately as **v0.3-only** flags and are not used to judge history.

**Known exchange events.** The 1 Oct 09:00–10:05 UTC window was Deribit's planned platform migration for the Coinbase International consolidation. It is labelled as a known event, so it can be excluded as a block rather than through threshold flags.

In [ ]:
ROBUST_Z = 8.0
LOG_SPREAD_FLOOR = 0.10            # scale floor for log spread (≈10% change)
BASIS_FLOOR_BPS = 1.0              # scale floor for basis
TICK_L2_FLOOR_BPS = 1.0

WIDE_SPREAD_MIN_BPS = 5.0          # economic floor: spread must be at least this wide
WIDE_SPREAD_MIN_RATIO = 10.0       # ...and at least this multiple of the asset's median spread
BASIS_MIN_DEVIATION_BPS = 10.0     # basis must move at least this far from its median
STALE_DRIFT_MIN_BPS = 5.0          # index drift needed before an unchanged BBO counts as stale
LAST_TRADE_OUTSIDE_MIN_BPS = 5.0

KNOWN_EXCHANGE_EVENTS = [
    # (start inclusive, end inclusive, UTC, label)
    ("2026-10-01 09:00", "2026-10-01 10:05",
     "Deribit platform migration (Coinbase International consolidation)"),
]

def robust_z(x, floor):
    x = pd.to_numeric(x, errors="coerce")
    med = x.median()
    mad = (x - med).abs().median()
    scale = max(1.4826 * mad, floor) if np.isfinite(mad) else floor
    return (x - med) / scale

def add_flags(g, z=ROBUST_Z):
    g = g.copy()
    spread = g["tick_spread_bps"]
    median_spread = spread.median()
    log_spread = np.log(spread.where(spread > 0))
    basis_dev = (g["mid_vs_index_bps"] - g["mid_vs_index_bps"].median()).abs()

    g["z_log_spread"] = robust_z(log_spread, LOG_SPREAD_FLOOR)
    g["z_basis"] = robust_z(g["mid_vs_index_bps"], BASIS_FLOOR_BPS)
    g["z_tick_l2"] = robust_z(g["tick_vs_l2_mid_bps"], TICK_L2_FLOOR_BPS)

    trades = num(g, "trade_count").fillna(0)
    updates = num(g, "book_update_count")
    is_v03 = g["collector_version"].notna() if "collector_version" in g.columns else pd.Series(False, index=g.index)

    g["flag_wide_spread"] = (
        (g["z_log_spread"] > z)
        & (spread >= WIDE_SPREAD_MIN_BPS)
        & (spread >= WIDE_SPREAD_MIN_RATIO * median_spread)
    )
    g["flag_mid_off_index"] = (g["z_basis"].abs() > z) & (basis_dev >= BASIS_MIN_DEVIATION_BPS)
    g["flag_stale_vs_index"] = (
        g["bbo_unchanged"]
        & (g["index_drift_since_bbo_change_bps"] > np.maximum(0.5 * spread, STALE_DRIFT_MIN_BPS))
    )
    g["flag_frozen_with_trades"] = g["bbo_frozen_trade_bars"] >= FROZEN_MIN_BARS
    g["flag_no_l2_updates"] = (updates == 0) & (trades > 0)
    g["flag_no_two_sided"] = ~g["two_sided_ticker"].astype(bool)

    # v0.3-only flags: evaluated only on stamped rows.
    g["flag_tick_l2_disagree"] = is_v03 & (g["z_tick_l2"] > z)
    g["flag_trade_outside_bbo"] = is_v03 & (g["last_trade_outside_bbo_bps"] > LAST_TRADE_OUTSIDE_MIN_BPS)

    g["known_exchange_event"] = False
    for start, end, _label in KNOWN_EXCHANGE_EVENTS:
        g.loc[
            g["bar_end"].between(pd.Timestamp(start, tz="UTC"), pd.Timestamp(end, tz="UTC")),
            "known_exchange_event",
        ] = True
    return g

FLAG_COLS_HISTORY = [
    "flag_wide_spread", "flag_mid_off_index", "flag_stale_vs_index",
    "flag_frozen_with_trades", "flag_no_l2_updates", "flag_no_two_sided",
]
FLAG_COLS_V03 = ["flag_tick_l2_disagree", "flag_trade_outside_bbo"]
FLAG_COLS = FLAG_COLS_HISTORY + FLAG_COLS_V03

integrity = per_asset(integrity, add_flags)
integrity[FLAG_COLS + ["known_exchange_event"]] = (
    integrity[FLAG_COLS + ["known_exchange_event"]].fillna(False).astype(bool)
)
integrity["suspect_quote"] = integrity[FLAG_COLS].any(axis=1) | integrity["known_exchange_event"]

flag_counts = (
    integrity
    .groupby("underlying")[FLAG_COLS + ["known_exchange_event", "suspect_quote"]]
    .sum()
    .assign(states=integrity.groupby("underlying").size())
)
flag_pct = (
    flag_counts[FLAG_COLS + ["known_exchange_event", "suspect_quote"]]
    .div(flag_counts["states"], axis=0)
    .mul(100)
)

print("Flagged states (count). v0.3-only flags are evaluated on stamped rows only:")
display(flag_counts)
print("Flagged states (% of raw states):")
display(flag_pct.round(3))

outside_event = integrity[~integrity["known_exchange_event"]]
print("Suspect states outside the known exchange event, by flag:")
display(outside_event.groupby("underlying")[FLAG_COLS_HISTORY].sum())

# Sensitivity of the robust-z flags to the cutoff (floors held fixed).
sensitivity_rows = []
for z in [4, 6, 8, 10, 15]:
    tmp = per_asset(integrity, add_flags, z=z)
    for asset, g in tmp.groupby("underlying"):
        sensitivity_rows.append({
            "underlying": asset, "z": z,
            "wide_spread": int(g["flag_wide_spread"].sum()),
            "mid_off_index": int(g["flag_mid_off_index"].sum()),
        })
flag_sensitivity = pd.DataFrame(sensitivity_rows)
display(flag_sensitivity.pivot(index="underlying", columns="z"))

## 28. Suspect episodes

Consecutive suspect states per asset are merged into **episodes**. A single long episode and many isolated states call for different gate designs: the first points to a feed failure, the second to a threshold or an illiquid asset.

`cross_asset_overlap` counts how many other underlyings have a suspect state at the same timestamp. Simultaneous episodes across unrelated perpetuals point to the collector or the exchange, not the market.

In [ ]:
work = integrity.sort_values(["underlying", "bar_end"]).copy()
prev_suspect = work.groupby("underlying")["suspect_quote"].shift(1, fill_value=False)
prev_time = work.groupby("underlying")["bar_end"].shift(1)
contiguous = (work["bar_end"] - prev_time) == pd.Timedelta(minutes=5)
work["_episode_start"] = work["suspect_quote"] & ~(prev_suspect & contiguous)
work["episode_id"] = work.groupby("underlying")["_episode_start"].cumsum().where(work["suspect_quote"])

suspect_by_time = work.loc[work["suspect_quote"]].groupby("bar_end")["underlying"].nunique()
work["cross_asset_overlap"] = work["bar_end"].map(suspect_by_time).fillna(0).astype(int) - work["suspect_quote"].astype(int)

ep = work.loc[work["suspect_quote"]]
episodes = (
    ep.groupby(["underlying", "episode_id"])
    .agg(
        start=("bar_end", "min"),
        end=("bar_end", "max"),
        states=("bar_end", "size"),
        trades=("trade_count", "sum"),
        volume_usd=("volume_usd", "sum"),
        max_spread_bps=("tick_spread_bps", "max"),
        max_abs_basis_bps=("mid_vs_index_bps", lambda s: s.abs().max()),
        max_abs_mid_return=("mid_return_5m_raw", lambda s: s.abs().max()),
        max_abs_index_return=("index_return_5m", lambda s: s.abs().max()),
        max_cross_asset_overlap=("cross_asset_overlap", "max"),
        known_event_states=("known_exchange_event", "sum"),
        **{f.replace("flag_", "n_"): (f, "sum") for f in FLAG_COLS},
    )
    .reset_index()
    .sort_values(["states", "max_spread_bps"], ascending=False)
)

print(f"Suspect states: {int(work['suspect_quote'].sum()):,} in {len(episodes):,} episodes")
display(
    episodes.groupby("underlying").agg(
        episodes=("states", "size"),
        single_state=("states", lambda s: int((s == 1).sum())),
        median_states=("states", "median"),
        max_states=("states", "max"),
    )
)
print("Longest / widest episodes:")
display(episodes.head(30))

print("Suspect states by UTC date and underlying:")
display(
    work.assign(date=work["bar_end"].dt.date)
    .pivot_table(index="date", columns="underlying", values="suspect_quote", aggfunc="sum", fill_value=0)
)

## 29. Known windows from the audit

The three windows found in Parts D–E are re-examined on raw fields. The plot windows are deliberately wide (±3 h): the audit times were read from notebook outputs printed before the UTC fix, so they may carry a two-hour offset.

For each window the question is the same: **did the market move, or did the quote stop updating?** The index and the trade VWAP are the references; the perpetual mid is the suspect.

In [ ]:
KNOWN_WINDOWS = [
    ("TRX ±9.6% round trip on zero volume", "2026-09-21 17:10", ["TRX", "AVAX", "XRP"]),
    ("HYPE / XRP wide book spreads, no trades", "2026-09-28 09:25", ["HYPE", "XRP"]),
    ("Deribit migration window: frozen BBO while trading", "2026-10-01 09:30", ["XRP", "HYPE", "TRX", "AVAX"]),
]
WINDOW_HALF_WIDTH = pd.Timedelta(hours=3)

window_cols = [
    "bar_end", "underlying", "tick_spread_bps", "l2_spread_bps", "mid_vs_index_bps",
    "vwap_vs_index_bps", "tick_vs_l2_mid_bps", "bbo_run_bars", "index_drift_since_bbo_change_bps",
    "last_trade_outside_bbo_bps", "trade_count", "volume_usd", "book_update_count",
    "book_age_ms", "ticker_age_ms", "mid_return_5m_raw", "index_return_5m",
    "stream_complete", "suspect_quote",
] + FLAG_COLS
window_cols = [c for c in window_cols if c in integrity.columns]

for title, centre, assets in KNOWN_WINDOWS:
    centre = pd.Timestamp(centre, tz="UTC")
    lo, hi = centre - WINDOW_HALF_WIDTH, centre + WINDOW_HALF_WIDTH
    w = integrity[(integrity["bar_end"].between(lo, hi)) & (integrity["underlying"].isin(assets))]

    print(f"\n=== {title} | {lo:%Y-%m-%d %H:%M} – {hi:%H:%M} UTC ===")
    if w.empty:
        print("No raw rows in this window.")
        continue

    display(
        w.groupby("underlying").agg(
            states=("bar_end", "size"),
            suspect=("suspect_quote", "sum"),
            first_suspect=("bar_end", lambda s: s[w.loc[s.index, "suspect_quote"]].min()),
            last_suspect=("bar_end", lambda s: s[w.loc[s.index, "suspect_quote"]].max()),
            max_spread_bps=("tick_spread_bps", "max"),
            trades=("trade_count", "sum"),
        )
    )
    display(w.loc[w["suspect_quote"], window_cols].head(40))

    fig, axes = plt.subplots(len(assets), 1, figsize=(13, 2.8 * len(assets)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, asset in zip(axes, assets):
        a = w[w["underlying"] == asset].sort_values("bar_end")
        if a.empty:
            ax.set_title(f"{asset}: no rows")
            continue
        ref = pd.to_numeric(a["index_price"], errors="coerce")
        base = ref.dropna().iloc[0] if ref.notna().any() else np.nan
        to_bps = lambda s: 1e4 * np.log(pd.to_numeric(s, errors="coerce") / base)
        ax.plot(a["bar_end"], to_bps(a["index_price"]), label="index", linewidth=1.6)
        ax.plot(a["bar_end"], to_bps(a["tick_mid"]), label="perp mid", linewidth=1.1)
        ax.fill_between(
            a["bar_end"], to_bps(a["best_bid_price"]), to_bps(a["best_ask_price"]),
            alpha=0.15, label="ticker bid–ask",
        )
        ax.scatter(a["bar_end"], to_bps(a["vwap"]), s=8, label="trade VWAP", zorder=3)
        sus = a[a["suspect_quote"]]
        ax.scatter(sus["bar_end"], np.zeros(len(sus)), marker="|", s=200, label="suspect", zorder=4)
        ax.set_title(asset)
        ax.set_ylabel("bps vs first index")
        ax.legend(loc="upper left", fontsize=8, ncol=5)
    axes[-1].set_xlabel("UTC")
    plt.tight_layout()
    plt.show()

## 30. Impact on returns and realized volatility

From market feature **v0.2.0**, `return_5m` and the realized-volatility columns are computed from the **Deribit index** (`return_source = 'deribit_index'`). The previous perpetual-mid return is kept as the diagnostic column `perp_mid_return_5m`.

Three series are compared on the same grid:

1. `perp_mid_return_5m`: the perpetual mid (the pre-v0.2.0 definition);
2. the same mid return with any return whose start or end state is suspect set to missing;
3. `return_5m`: the index return now used by the pipeline.

The recomputed raw mid and index returns are first checked against the pipeline columns, so the comparison is between data sources and not between implementations.

What to look for: the index should keep the genuine tails and median volatility while losing the isolated spikes. Masking the mid should **not** be the fix, because any flag correlated with volatility censors the high-volatility bars the regime model needs.

In [ ]:
has_perp_col = "perp_mid_return_5m" in features.columns
pipeline_source = (
    features["return_source"].dropna().unique().tolist()
    if "return_source" in features.columns else ["perp_mid (pre-v0.2.0 table)"]
)
print(f"Pipeline return_source: {pipeline_source}")
if not has_perp_col:
    print("perp_mid_return_5m not found: the feature table predates v0.2.0. "
          "Rebuild with build_features.py before reading this section.")

feature_cols = ["underlying", "state_time", "return_5m", "realized_vol_30m_annualized"]
if has_perp_col:
    feature_cols.append("perp_mid_return_5m")

impact = integrity.merge(
    features[feature_cols].assign(state_time=lambda d: pd.to_datetime(d["state_time"], utc=True)),
    left_on=["underlying", "bar_end"],
    right_on=["underlying", "state_time"],
    how="inner",
).sort_values(["underlying", "bar_end"])

mid_col = "perp_mid_return_5m" if has_perp_col else "return_5m"
index_col = "return_5m" if has_perp_col else "index_return_5m"

# 1) Implementation checks against the pipeline.
for label, pipe, raw in [
    ("perp mid", mid_col, "mid_return_5m_raw"),
    ("index", index_col, "index_return_5m"),
]:
    both = impact[[pipe, raw]].dropna()
    print(
        f"{label:<9} pairs: {len(both):>7,} | max |pipeline − raw|: "
        f"{(both[pipe] - both[raw]).abs().max():.3e} | "
        f"pipeline values without raw value: {int(impact[pipe].notna().sum() - len(both)):,}"
    )

# 2) Mask mid returns touching a suspect endpoint.
impact = per_asset(impact, lambda g: g.assign(_prev_suspect=lag_on_grid(g, "suspect_quote")))
prev_suspect = impact["_prev_suspect"].fillna(False).astype(bool)
impact["mid_return_5m_clean"] = impact[mid_col].where(~(impact["suspect_quote"] | prev_suspect))

PERIODS_PER_YEAR = 365.0 * 24.0 * 12.0

def rv_annualized(r, window=6):
    """Complete-window RV, same definition as the pipeline (requires `window` returns)."""
    s = r.pow(2).rolling(window, min_periods=window).sum()
    return np.sqrt(s / window * PERIODS_PER_YEAR)

impact["rv30_mid"] = impact.groupby("underlying", group_keys=False)[mid_col].apply(rv_annualized)
impact["rv30_mid_clean"] = impact.groupby("underlying", group_keys=False)["mid_return_5m_clean"].apply(rv_annualized)
impact["rv30_index"] = impact.groupby("underlying", group_keys=False)[index_col].apply(rv_annualized)

rows, pair_rows = [], []
for asset, g in impact.groupby("underlying"):
    for label, col in [
        ("perp mid", mid_col),
        ("perp mid, suspect masked", "mid_return_5m_clean"),
        ("index (pipeline)", index_col),
    ]:
        s = g[col].dropna()
        rows.append({
            "underlying": asset,
            "series": label,
            "n": len(s),
            "zero_share_pct": 100 * (s == 0).mean() if len(s) else np.nan,
            "std_bps": 1e4 * s.std(),
            "excess_kurtosis": s.kurt(),
            "max_abs_bps": 1e4 * s.abs().max() if len(s) else np.nan,
        })

    pair = g[[mid_col, index_col]].dropna()
    pair_rows.append({
        "underlying": asset,
        "corr_mid_index": pair.corr().iloc[0, 1] if len(pair) > 20 else np.nan,
        "max_rv30_mid": g["rv30_mid"].max(),
        "max_rv30_mid_masked": g["rv30_mid_clean"].max(),
        "max_rv30_index": g["rv30_index"].max(),
        "median_rv30_mid": g["rv30_mid"].median(),
        "median_rv30_index": g["rv30_index"].median(),
        "masked_share_pct": 100 * (g[mid_col].notna() & g["mid_return_5m_clean"].isna()).mean(),
    })

return_source_impact = pd.DataFrame(rows)
return_source_pairs = pd.DataFrame(pair_rows)
print("Return distribution by source:")
display(return_source_impact.pivot(index="underlying", columns="series").round(3))
print("Mid vs index agreement and 30m RV (annualized):")
display(return_source_pairs.round(4))

for asset, g in impact.groupby("underlying"):
    pair = g[[mid_col, index_col, "suspect_quote"]].dropna(subset=[mid_col, index_col])
    if pair.empty:
        continue
    fig, ax = plt.subplots(figsize=(5.2, 5.2))
    ax.scatter(1e4 * pair[index_col], 1e4 * pair[mid_col], s=4, alpha=0.4, label="clean")
    s = pair[pair["suspect_quote"]]
    ax.scatter(1e4 * s[index_col], 1e4 * s[mid_col], s=14, color="red", label="suspect end-state")
    lim = np.nanmax(np.abs(1e4 * pair[[mid_col, index_col]].to_numpy()))
    ax.plot([-lim, lim], [-lim, lim], linewidth=0.8)
    ax.set_xlabel("index return 5m (bps)")
    ax.set_ylabel("perp-mid return 5m (bps)")
    ax.set_title(asset)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 31. Open-interest jumps

The first run of this section rejected the sawtooth hypothesis (OI returns are about half negative, median ≈ 0) and attributed 9 of the 16 jumps above 10% to **09:05 and 09:20 UTC on 1 Oct**. That is the Deribit migration window: positions from Coinbase International were transferred onto Deribit's perpetuals. Those OI steps are real positions but not trading activity, and the new levels persisted.

Consequences tested below:

1. **Version boundary.** OI levels before and after `OI_VERSION_BOUNDARY` are not comparable. OI changes inside the migration window are excluded.
2. **Remaining jumps.** Jumps outside the migration are classified as one-bar reversals (`|next return| ≥ 80%` of the jump, opposite sign) or persistent level shifts.
3. **Static OI.** An asset whose OI is unchanged in most bars carries no OI information.

The table at the end gives a per-asset OI usability verdict. It is evidence for the feature contract, not a decision.

In [ ]:
OI_JUMP_THRESHOLD = 0.10
OI_VERSION_BOUNDARY = pd.Timestamp("2026-10-01 10:05", tz="UTC")
OI_STATIC_MAX_SHARE = 0.25          # OI unchanged in more than this share of bars -> uninformative

oi = raw_futures_all.sort_values(["underlying", "bar_end"]).copy()
oi["open_interest"] = pd.to_numeric(oi["open_interest"], errors="coerce")
oi = per_asset(oi, lambda g: g.assign(_prev_oi=lag_on_grid(g, "open_interest")))
oi["oi_return_5m"] = (oi["open_interest"] / oi["_prev_oi"] - 1).where(oi["_prev_oi"] > 0)
oi["_next_oi_return"] = oi.groupby("underlying")["oi_return_5m"].shift(-1)
oi["_prev_instrument"] = oi.groupby("underlying")["instrument_name"].shift(1)
oi["instrument_changed"] = oi["instrument_name"] != oi["_prev_instrument"]
oi["utc_hhmm"] = oi["bar_end"].dt.strftime("%H:%M")

event_mask = pd.Series(False, index=oi.index)
for start, end, _label in KNOWN_EXCHANGE_EVENTS:
    event_mask |= oi["bar_end"].between(pd.Timestamp(start, tz="UTC"), pd.Timestamp(end, tz="UTC"))
oi["in_known_event"] = event_mask
oi["oi_regime"] = np.where(oi["bar_end"] > OI_VERSION_BOUNDARY, "post-migration", "pre-migration")

jumps = oi[oi["oi_return_5m"].abs() > OI_JUMP_THRESHOLD].copy()
reverts = (
    (np.sign(jumps["_next_oi_return"]) == -np.sign(jumps["oi_return_5m"]))
    & (jumps["_next_oi_return"].abs() >= 0.8 * jumps["oi_return_5m"].abs() / (1 + jumps["oi_return_5m"]))
)
jumps["jump_type"] = np.select(
    [jumps["in_known_event"], reverts],
    ["migration (known event)", "one-bar reversal"],
    default="persistent level shift",
)

jump_cols = [
    "underlying", "bar_end", "utc_hhmm", "jump_type", "instrument_changed",
    "_prev_oi", "open_interest", "oi_return_5m", "_next_oi_return",
    "stream_complete", "ticker_age_ms", "trade_count", "volume_usd",
]
oi_jumps = jumps[[c for c in jump_cols if c in jumps.columns]].sort_values("bar_end")
print(f"OI jumps with |return| > {OI_JUMP_THRESHOLD:.0%}: {len(oi_jumps):,}")
display(oi_jumps["jump_type"].value_counts().to_frame("jumps"))
display(oi_jumps)

# Level break at the migration.
level = (
    oi[~oi["in_known_event"]]
    .groupby(["underlying", "oi_regime"])["open_interest"]
    .median()
    .unstack()
)
if {"pre-migration", "post-migration"}.issubset(level.columns):
    level["post_over_pre"] = level["post-migration"] / level["pre-migration"]
print("Median OI before and after the migration boundary:")
display(level.round(3))

# Per-asset usability evidence (migration window excluded).
clean = oi[~oi["in_known_event"]]
usability = clean.groupby("underlying").agg(
    bars=("oi_return_5m", "count"),
    static_share=("oi_return_5m", lambda s: (s == 0).mean()),
    negative_share=("oi_return_5m", lambda s: (s < 0).mean()),
    abs_ret_p99_pct=("oi_return_5m", lambda s: 100 * s.abs().quantile(0.99)),
)
usability = usability.join(
    jumps[~jumps["in_known_event"]]
    .groupby("underlying")["jump_type"]
    .value_counts()
    .unstack(fill_value=0),
    how="left",
).fillna(0)
usability["oi_informative"] = usability["static_share"] <= OI_STATIC_MAX_SHARE
display(usability.round(4))

assets = sorted(oi["underlying"].dropna().unique())
fig, axes = plt.subplots(len(assets), 1, figsize=(13, 2.3 * len(assets)), sharex=True)
for ax, asset in zip(np.atleast_1d(axes), assets):
    a = oi[oi["underlying"] == asset]
    ax.plot(a["bar_end"], a["open_interest"], linewidth=0.9)
    j = jumps[jumps["underlying"] == asset]
    ax.scatter(j["bar_end"], j["open_interest"], color="red", s=12, zorder=3)
    ax.axvline(OI_VERSION_BOUNDARY, linestyle="--", linewidth=0.8)
    ax.set_title(f"{asset} open interest (red: |Δ| > {OI_JUMP_THRESHOLD:.0%}; dashed: migration boundary)", fontsize=10)
np.atleast_1d(axes)[-1].set_xlabel("UTC")
plt.tight_layout()
plt.show()

## 32. Integrity findings to carry forward

`futures_integrity_diagnostics` (in memory, keyed by `underlying × state_time`) holds every metric and flag from this part. It is **not** written to the database.

Findings from the first full run (2026-09-21 → 2026-10-05), to be re-checked after each rebuild:

1. **All three audit windows are data defects.** 1 Oct 09:00–10:05 (all 7 assets) was Deribit's planned migration. 21 Sep 17:10 (AVAX/TRX/XRP, one snapshot) and 28 Sep 09:05–09:30 (collector outage, then wide alt quotes) are unexplained feed events affecting several assets at once.
2. **Structural returns moved to the index (feature v0.2.0).** On the perpetual mid, TRX return kurtosis was 1,508 and its correlation with the index 0.21. The index removes these artifacts and leaves median RV unchanged.
3. **Masking is not a substitute.** A flag correlated with volatility censors genuine high-volatility bars.
4. **Flags for the production gate** (to implement in `quality_contract.py` with training-only thresholds): frozen BBO with trades, no L2 updates with trades, no two-sided quote, mid off index with an absolute floor, wide spread with an absolute and relative floor, and known exchange events.
5. **Open interest** has a level break at the 1 Oct migration and is static for TRX and HYPE. See §31 for the per-asset verdict.

Post-v0.3 rows carry direct signals (`bbo_last_change`, `trades_outside_book_bbo`, REST stale confirmation). Once a few weeks exist, compare them with the pre-v0.3 proxies above on the overlapping period.

In [ ]:
futures_integrity_diagnostics = integrity.rename(columns={"bar_end": "state_time"})[
    ["underlying", "state_time", "suspect_quote", "known_exchange_event"]
    + FLAG_COLS
    + [c for c in metric_cols if c in integrity.columns]
    + ["index_return_5m", "mid_return_5m_raw"]
]

print(f"Rows:                 {len(futures_integrity_diagnostics):,}")
print(f"Suspect rows:         {int(futures_integrity_diagnostics['suspect_quote'].sum()):,}")
print(f"Known-event rows:     {int(futures_integrity_diagnostics['known_exchange_event'].sum()):,}")
print(f"Episodes:             {len(episodes):,}")
print(f"OI jumps:             {len(oi_jumps):,}")

# Part J — Research Handoff

## 33. Notebook 02 interpretation checklist

Before moving to `03_surface_distributions.ipynb`, record the answers to these questions:

1. Which market features show materially different scale across underlyings?
2. Which features have genuine structural boundary masses rather than smooth distributions?
3. Which tails are driven by isolated observations versus repeated episodes?
4. Are extreme spreads/basis observations concentrated in particular assets or market conditions?
5. Which variables show enough persistence to be plausible structural-state descriptors?
6. Which variables change too rapidly and may be better suited to the future noise/transition blocks?
7. Does `log1p_volume_usd` materially improve the activity representation relative to raw volume?
8. Which positive scale variables are plausible `log1p` candidates?
9. Which signed scale variables are plausible signed-log candidates?
10. Which bounded variables should remain in native support?
11. Is session-to-session distribution drift large enough to affect later chronological validation design?
12. Are any findings based on too little history to support a production decision?
13. Which tails in Parts A–E survive once suspect quote states are masked (Section 30)?
14. Should structural returns move from the perpetual mid to the index (audit A1), given Section 30?
15. Which integrity flags belong in the production gate, and which never fire (Section 32)?
16. What explains the open-interest jumps, and can OI features enter the contract (Section 31)?

**Do not freeze the model feature contract here.**  
Notebook 02 creates evidence for the later feature-contract decision; it does not make the final decision.

In [ ]:
market_distribution_report = {
    "overall_availability": overall_availability,
    "availability_by_asset": availability_by_asset,
    "return_pooled": return_pooled if "return_pooled" in globals() else pd.DataFrame(),
    "return_by_asset": return_by_asset if "return_by_asset" in globals() else pd.DataFrame(),
    "rv_summary": rv_summary if "rv_summary" in globals() else pd.DataFrame(),
    "rv_by_asset": rv_by_asset if "rv_by_asset" in globals() else pd.DataFrame(),
    "boundary_mass": boundary_mass if "boundary_mass" in globals() else pd.DataFrame(),
    "cross_asset_scale": cross_asset_scale if "cross_asset_scale" in globals() else pd.DataFrame(),
    "session_distribution": session_distribution if "session_distribution" in globals() else pd.DataFrame(),
    "transform_assessment": transform_assessment if "transform_assessment" in globals() else pd.DataFrame(),
    "integrity_flag_counts": flag_counts.reset_index() if "flag_counts" in globals() else pd.DataFrame(),
    "integrity_flag_sensitivity": flag_sensitivity if "flag_sensitivity" in globals() else pd.DataFrame(),
    "integrity_episodes": episodes if "episodes" in globals() else pd.DataFrame(),
    "return_source_impact": return_source_impact if "return_source_impact" in globals() else pd.DataFrame(),
    "return_source_pairs": return_source_pairs if "return_source_pairs" in globals() else pd.DataFrame(),
    "oi_jumps": oi_jumps if "oi_jumps" in globals() else pd.DataFrame(),
    "futures_integrity_diagnostics": futures_integrity_diagnostics if "futures_integrity_diagnostics" in globals() else pd.DataFrame(),
}

print("Market-distribution report sections:")
for name, frame in market_distribution_report.items():
    print(f"  {name:<28} {len(frame):>7,} rows")

## 34. Close the read-only connection

In [ ]:
con.close()
print("DuckDB connection closed.")